# Optimal Estimation Demo

Scientific question: can the current Python forward model and O2 A Jacobians drive a two-state Rodgers-style optimal-estimation retrieval for surface albedo and aerosol optical depth?

This is an executable retrieval-flow demo, not a new public retrieval engine. It builds a synthetic measurement, forms a two-state vector, iterates an OE update, and writes convergence history under `out/demo/optimal_estimation_demo/`.

## Inputs and API Calls

The state vector is `[surface_albedo, aerosol_optical_depth_550_nm]`. Each iteration prepares an O2 A case, runs `forward_model(jacobian=True)`, converts radiance Jacobians to reflectance Jacobians, and solves the prior-regularized linear OE step.

In [ ]:
from __future__ import annotations

import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "build.zig").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repository root")


REPO_ROOT = find_repo_root()
PYTHON_ROOT = REPO_ROOT / "python"
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(PYTHON_ROOT))

import zdisamar as zd  # noqa: E402

from validation.common.o2a_reference_case import build_o2a_case  # noqa: E402

OUT_DIR = REPO_ROOT / "out" / "demo" / "optimal_estimation_demo"
LIBRARY_NAME = "libzdisamar_c.dylib" if sys.platform == "darwin" else "libzdisamar_c.so"
LIBRARY_PATH = REPO_ROOT / "zig-out" / "lib" / LIBRARY_NAME
STATE_NAMES = ("surface_albedo", "aerosol_optical_depth")


def require_library() -> str:
    if not LIBRARY_PATH.exists():
        raise FileNotFoundError(
            f"{LIBRARY_PATH} does not exist; build the native shared library first"
        )
    return str(LIBRARY_PATH)


def make_case(state: np.ndarray):
    case = build_o2a_case(zd)
    case.spectral_grid.sample_count = 81
    case.surface.albedo = float(state[0])
    case.aerosol.optical_depth_550_nm = float(state[1])
    return case


def run_reflectance(case, *, jacobian: bool) -> tuple[np.ndarray, np.ndarray, np.ndarray | None]:
    library = require_library()
    with (
        zd.prepare(case, library_path=library) as prepared,
        prepared.forward_model(jacobian=jacobian) as spectrum,
    ):
        wavelength_nm = spectrum.wavelength_nm.copy()
        reflectance = spectrum.reflectance.copy()
        if not jacobian:
            return wavelength_nm, reflectance, None
        state_indexes = [spectrum.jacobian_state_names.index(name) for name in STATE_NAMES]
        radiance_jacobian = spectrum.radiance_jacobian.copy()[:, state_indexes]
        irradiance = spectrum.irradiance.copy()
    mu0 = math.cos(math.radians(case.geometry.solar_zenith_deg))
    reflectance_jacobian = radiance_jacobian / ((mu0 * irradiance / math.pi)[:, None])
    return wavelength_nm, reflectance, reflectance_jacobian


def optimal_estimation_step(
    x: np.ndarray, xa: np.ndarray, y: np.ndarray, se: np.ndarray, sa: np.ndarray
):
    case = make_case(x)
    wavelength_nm, modeled, jacobian = run_reflectance(case, jacobian=True)
    if jacobian is None:
        raise RuntimeError("Jacobian was not returned")
    residual = y - modeled
    se_inv = np.diag(1.0 / np.square(se))
    sa_inv = np.diag(1.0 / np.square(sa))
    hessian = jacobian.T @ se_inv @ jacobian + sa_inv
    gradient = jacobian.T @ se_inv @ residual - sa_inv @ (x - xa)
    dx = np.linalg.solve(hessian, gradient)
    cost = float(residual.T @ se_inv @ residual + (x - xa).T @ sa_inv @ (x - xa))
    return wavelength_nm, modeled, jacobian, residual, dx, cost


def run_optimal_estimation_demo() -> dict[str, object]:
    truth = np.array([0.18, 0.38], dtype=float)
    prior = np.array([0.24, 0.22], dtype=float)
    prior_sigma = np.array([0.08, 0.18], dtype=float)
    x = prior.copy()

    measurement_case = make_case(truth)
    wavelength_nm, measurement, _ = run_reflectance(measurement_case, jacobian=False)
    measurement_sigma = np.full_like(measurement, 4.0e-4)

    rows: list[dict[str, float | int]] = []
    start = time.perf_counter()
    for iteration in range(6):
        _, modeled, jacobian, residual, dx, cost = optimal_estimation_step(
            x,
            prior,
            measurement,
            measurement_sigma,
            prior_sigma,
        )
        rows.append(
            {
                "iteration": iteration,
                "surface_albedo": float(x[0]),
                "aerosol_optical_depth": float(x[1]),
                "cost": cost,
                "max_abs_residual": float(np.max(np.abs(residual))),
                "step_norm": float(np.linalg.norm(dx)),
            }
        )
        x = np.array([np.clip(x[0] + dx[0], 0.02, 0.8), np.clip(x[1] + dx[1], 0.01, 1.5)])
        if np.linalg.norm(dx / prior_sigma) < 1.0e-3:
            break

    _, final_modeled, final_jacobian, final_residual, _, final_cost = optimal_estimation_step(
        x,
        prior,
        measurement,
        measurement_sigma,
        prior_sigma,
    )
    history = pd.DataFrame(rows)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    history_path = OUT_DIR / "history.csv"
    spectrum_path = OUT_DIR / "measurement_fit.csv"
    summary_path = OUT_DIR / "summary.json"
    history.to_csv(history_path, index=False)
    pd.DataFrame(
        {
            "wavelength_nm": wavelength_nm,
            "measurement_reflectance": measurement,
            "modeled_reflectance": final_modeled,
            "residual": final_residual,
            "surface_albedo_jacobian": final_jacobian[:, 0],
            "aerosol_optical_depth_jacobian": final_jacobian[:, 1],
        }
    ).to_csv(spectrum_path, index=False)
    summary = {
        "state_names": list(STATE_NAMES),
        "prior_state": prior.tolist(),
        "truth_state": truth.tolist(),
        "retrieved_state": x.tolist(),
        "final_cost": final_cost,
        "iterations": len(rows),
        "final_max_abs_residual": float(np.max(np.abs(final_residual))),
        "timing": {"total_s": time.perf_counter() - start},
        "artifacts": {
            "history_csv": str(history_path),
            "measurement_fit_csv": str(spectrum_path),
            "summary_json": str(summary_path),
        },
    }
    summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    return summary

## Execute and Interpret

The retrieved aerosol state, convergence history, measurement fit, and residuals are written as disposable demo outputs.

In [ ]:
summary = run_optimal_estimation_demo()
history = pd.read_csv(summary["artifacts"]["history_csv"])
summary, history.tail(3)